# Exploratory Data Analysis — MedRisk Readmission Prediction

This notebook explores the UCI Diabetes 130-US Hospitals dataset to understand feature distributions, missing values, class imbalance, and relationships with the 30-day readmission target.

In [ ]:
%matplotlib inline

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

## 1. Dataset Overview

In [ ]:
try:
    from src.data.loader import load_raw_data
    df = load_raw_data()
    print("Loaded full dataset.")
except Exception:
    df = pd.read_csv("../data/sample/sample_data.csv", na_values=["?"])
    print("Full dataset not available — using sample data.")

print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes.value_counts()}")

In [ ]:
df.head()

In [ ]:
df.describe()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["count"] > 0].sort_values("pct", ascending=False)
print(f"Columns with missing values: {len(missing_df)}")
missing_df

In [ ]:
cols_with_missing = missing_df.index.tolist()
if cols_with_missing:
    fig, ax = plt.subplots(figsize=(12, max(4, len(cols_with_missing) * 0.4)))
    sns.heatmap(
        df[cols_with_missing].isnull().T,
        cbar=False,
        yticklabels=True,
        cmap="YlOrRd",
        ax=ax,
    )
    ax.set_title("Missing Value Heatmap")
    ax.set_xlabel("Sample Index")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected (or already encoded as '?').")

## 3. Target Distribution

In [ ]:
target_counts = df["readmitted"].value_counts()
target_pct = df["readmitted"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(target_counts.index.astype(str), target_counts.values, color=["#2ecc71", "#e74c3c", "#f39c12"])
for bar, pct in zip(bars, target_pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + len(df) * 0.01,
        f"{pct:.1f}%",
        ha="center",
        fontweight="bold",
    )
ax.set_title("Readmission Target Distribution")
ax.set_xlabel("Readmitted")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

readmit_30 = (df["readmitted"] == "<30").sum()
print(f"\n<30 day readmission rate: {readmit_30 / len(df) * 100:.1f}% ({readmit_30:,} / {len(df):,})")

## 4. Feature Distributions

In [ ]:
numeric_features = [
    "time_in_hospital", "num_lab_procedures", "num_medications",
    "number_diagnoses", "number_inpatient", "number_emergency", "number_outpatient",
]
available = [c for c in numeric_features if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(available):
    ax = axes[i]
    df[col].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col, fontsize=11)
    ax.set_ylabel("Count")
for j in range(len(available), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Numeric Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Categorical Features

In [ ]:
cat_features = ["race", "gender", "age"]
available_cat = [c for c in cat_features if c in df.columns]

fig, axes = plt.subplots(1, len(available_cat), figsize=(5 * len(available_cat), 5))
if len(available_cat) == 1:
    axes = [axes]
for ax, col in zip(axes, available_cat):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette="Set2")
    ax.set_title(f"{col} Distribution")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Readmission Rate by Demographics

In [ ]:
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["race", "gender", "age"]):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    rates = df.groupby(col)["readmit_30"].mean().sort_values(ascending=False) * 100
    rates.plot(kind="bar", ax=ax, color="coral", edgecolor="white")
    ax.set_title(f"30-Day Readmission Rate by {col.title()}")
    ax.set_ylabel("Readmission Rate (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.axhline(df["readmit_30"].mean() * 100, color="black", linestyle="--", linewidth=1, label="Overall")
    ax.legend()

plt.tight_layout()
plt.show()

df.drop(columns=["readmit_30"], inplace=True)

## 7. Correlation Matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()